In [1]:
from pathlib import Path
from io import StringIO

import requests
import pandas as pd
import numpy as np

In [2]:
WEATHER_DIR = Path("data/external/weather")
RAW_DIR = WEATHER_DIR / "raw"

RAW_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = WEATHER_DIR / "toronto_pearson_weather_daily.parquet"

In [3]:
STATIONS = {
    "pearson_old": {
        "station_id": 5097,
        "start_year": 2011,
        "end_year": 2013
    },

    "pearson_current": {
        "station_id": 51459,
        "start_year": 2013,
        "end_year": 2026
    }
}

In [4]:
def download_daily_weather(station_id, year):

    url = (
        "https://climate.weather.gc.ca/climate_data/"
        "bulk_data_e.html"
        f"?format=csv"
        f"&stationID={station_id}"
        f"&Year={year}"
        f"&Month=1"
        f"&Day=1"
        f"&timeframe=2"
        f"&submit=Download+Data"
    )

    response = requests.get(
        url,
        timeout=60
    )

    response.raise_for_status()

    df = pd.read_csv(
        StringIO(response.text)
    )

    return df

In [5]:
test = download_daily_weather(
    station_id=51459,
    year=2025
)

print(test.shape)
print(test.columns.tolist())

display(test.head())

(365, 31)
['Longitude (x)', 'Latitude (y)', 'Station Name', 'Climate ID', 'Date/Time', 'Year', 'Month', 'Day', 'Data Quality', 'Max Temp (°C)', 'Max Temp Flag', 'Min Temp (°C)', 'Min Temp Flag', 'Mean Temp (°C)', 'Mean Temp Flag', 'Heat Deg Days (°C)', 'Heat Deg Days Flag', 'Cool Deg Days (°C)', 'Cool Deg Days Flag', 'Total Rain (mm)', 'Total Rain Flag', 'Total Snow (cm)', 'Total Snow Flag', 'Total Precip (mm)', 'Total Precip Flag', 'Snow on Grnd (cm)', 'Snow on Grnd Flag', 'Dir of Max Gust (10s deg)', 'Dir of Max Gust Flag', 'Spd of Max Gust (km/h)', 'Spd of Max Gust Flag']


,Longitude (x),Latitude (y),Station Name,Climate ID,Date/Time,Year,Month,Day,Data Quality,Max Temp (°C),...,Total Snow (cm),Total Snow Flag,Total Precip (mm),Total Precip Flag,Snow on Grnd (cm),Snow on Grnd Flag,Dir of Max Gust (10s deg),Dir of Max Gust Flag,Spd of Max Gust (km/h),Spd of Max Gust Flag
0,-79.63,43.68,TORONTO INTL A,6158731,2025-01-01,2025,1,1,NaN,2.8,...,0.0,T,0.7,NaN,NaN,NaN,30.0,NaN,62.0,NaN
1,-79.63,43.68,TORONTO INTL A,6158731,2025-01-02,2025,1,2,NaN,0.4,...,1.8,NaN,1.8,NaN,0.0,T,26.0,NaN,59.0,NaN
2,-79.63,43.68,TORONTO INTL A,6158731,2025-01-03,2025,1,3,NaN,-0.3,...,0.0,T,0.0,T,1.0,NaN,29.0,NaN,60.0,NaN
3,-79.63,43.68,TORONTO INTL A,6158731,2025-01-04,2025,1,4,NaN,-1.8,...,0.0,T,0.0,T,0.0,T,27.0,NaN,71.0,NaN
4,-79.63,43.68,TORONTO INTL A,6158731,2025-01-05,2025,1,5,NaN,-0.8,...,0.0,T,0.0,T,0.0,T,26.0,NaN,50.0,NaN


In [6]:
frames = []

for station_name, info in STATIONS.items():

    for year in range(
        info["start_year"],
        info["end_year"] + 1
    ):

        print(
            "Downloading:",
            station_name,
            year
        )

        try:

            df = download_daily_weather(
                info["station_id"],
                year
            )

            if len(df) == 0:
                print("No data")
                continue

            df["source_station_key"] = station_name
            df["source_station_id"] = info["station_id"]

            frames.append(df)

        except Exception as e:

            print(
                "FAILED:",
                station_name,
                year,
                e
            )

Downloading: pearson_old 2011
Downloading: pearson_old 2012
Downloading: pearson_old 2013
Downloading: pearson_current 2013
Downloading: pearson_current 2014
Downloading: pearson_current 2015
Downloading: pearson_current 2016
Downloading: pearson_current 2017
Downloading: pearson_current 2018
Downloading: pearson_current 2019
Downloading: pearson_current 2020
Downloading: pearson_current 2021
Downloading: pearson_current 2022
Downloading: pearson_current 2023
Downloading: pearson_current 2024
Downloading: pearson_current 2025
Downloading: pearson_current 2026


In [7]:
weather_raw = pd.concat(
    frames,
    ignore_index=True
)

print(weather_raw.shape)

display(weather_raw.head())

(6209, 33)


,Longitude (x),Latitude (y),Station Name,Climate ID,Date/Time,Year,Month,Day,Data Quality,Max Temp (°C),...,Total Precip (mm),Total Precip Flag,Snow on Grnd (cm),Snow on Grnd Flag,Dir of Max Gust (10s deg),Dir of Max Gust Flag,Spd of Max Gust (km/h),Spd of Max Gust Flag,source_station_key,source_station_id
0,-79.63,43.68,TORONTO LESTER B. PEARSON INT'L A,6158733,2011-01-01,2011,1,1,NaN,12.0,...,8.6,NaN,0.0,NaN,27.0,NaN,52,NaN,pearson_old,5097
1,-79.63,43.68,TORONTO LESTER B. PEARSON INT'L A,6158733,2011-01-02,2011,1,2,NaN,-1.1,...,0.0,T,0.0,NaN,26.0,NaN,39,NaN,pearson_old,5097
2,-79.63,43.68,TORONTO LESTER B. PEARSON INT'L A,6158733,2011-01-03,2011,1,3,NaN,0.1,...,0.0,T,0.0,NaN,25.0,NaN,32,NaN,pearson_old,5097
3,-79.63,43.68,TORONTO LESTER B. PEARSON INT'L A,6158733,2011-01-04,2011,1,4,NaN,1.3,...,0.4,NaN,0.0,T,NaN,NaN,<31,NaN,pearson_old,5097
4,-79.63,43.68,TORONTO LESTER B. PEARSON INT'L A,6158733,2011-01-05,2011,1,5,NaN,-1.0,...,0.0,NaN,0.0,T,30.0,NaN,33,NaN,pearson_old,5097


In [8]:
display(
    weather_raw[
        [
            "source_station_id",
            "Station Name"
        ]
    ]
    .drop_duplicates()
)

,source_station_id,Station Name
0,5097,TORONTO LESTER B. PEARSON INT'L A
1096,51459,TORONTO INTL A


In [9]:
weather_raw["date"] = pd.to_datetime(
    weather_raw["Date/Time"],
    errors="coerce"
)

In [10]:
print(
    weather_raw["date"].min(),
    "→",
    weather_raw["date"].max()
)

2011-01-01 00:00:00 → 2026-12-31 00:00:00


In [21]:
display(
    weather[
        weather["date"] > "2026-09-11"
    ].head(10)
)

,date,source_station_id,station_name,max_temperature,min_temperature,mean_temperature,total_rain,total_snow,total_precipitation,year,month,day_of_year,is_rain,is_snow
5733,2026-09-12,51459,TORONTO INTL A,NaN,NaN,NaN,NaN,NaN,NaN,2026,9,255,0,0
5734,2026-09-13,51459,TORONTO INTL A,NaN,NaN,NaN,NaN,NaN,NaN,2026,9,256,0,0
5735,2026-09-14,51459,TORONTO INTL A,NaN,NaN,NaN,NaN,NaN,NaN,2026,9,257,0,0
5736,2026-09-15,51459,TORONTO INTL A,NaN,NaN,NaN,NaN,NaN,NaN,2026,9,258,0,0
5737,2026-09-16,51459,TORONTO INTL A,NaN,NaN,NaN,NaN,NaN,NaN,2026,9,259,0,0
5738,2026-09-17,51459,TORONTO INTL A,NaN,NaN,NaN,NaN,NaN,NaN,2026,9,260,0,0
5739,2026-09-18,51459,TORONTO INTL A,NaN,NaN,NaN,NaN,NaN,NaN,2026,9,261,0,0
5740,2026-09-19,51459,TORONTO INTL A,NaN,NaN,NaN,NaN,NaN,NaN,2026,9,262,0,0
5741,2026-09-20,51459,TORONTO INTL A,NaN,NaN,NaN,NaN,NaN,NaN,2026,9,263,0,0
5742,2026-09-21,51459,TORONTO INTL A,NaN,NaN,NaN,NaN,NaN,NaN,2026,9,264,0,0


In [22]:
today = pd.Timestamp("2026-09-11")

weather = weather[
    weather["date"] <= today
].copy()

In [23]:
weather = weather[
    weather[
        [
            "mean_temperature",
            "min_temperature",
            "max_temperature",
            "total_precipitation"
        ]
    ].notna().any(axis=1)
].copy()

In [24]:
print(
    weather["date"].min(),
    "→",
    weather["date"].max()
)

2011-01-01 00:00:00 → 2026-09-10 00:00:00


In [11]:
for col in weather_raw.columns:
    print(repr(col))

'Longitude (x)'
'Latitude (y)'
'Station Name'
'Climate ID'
'Date/Time'
'Year'
'Month'
'Day'
'Data Quality'
'Max Temp (°C)'
'Max Temp Flag'
'Min Temp (°C)'
'Min Temp Flag'
'Mean Temp (°C)'
'Mean Temp Flag'
'Heat Deg Days (°C)'
'Heat Deg Days Flag'
'Cool Deg Days (°C)'
'Cool Deg Days Flag'
'Total Rain (mm)'
'Total Rain Flag'
'Total Snow (cm)'
'Total Snow Flag'
'Total Precip (mm)'
'Total Precip Flag'
'Snow on Grnd (cm)'
'Snow on Grnd Flag'
'Dir of Max Gust (10s deg)'
'Dir of Max Gust Flag'
'Spd of Max Gust (km/h)'
'Spd of Max Gust Flag'
'source_station_key'
'source_station_id'
'date'


In [12]:
weather = weather_raw[
    [
        "date",
        "source_station_id",
        "Station Name",
        "Max Temp (°C)",
        "Min Temp (°C)",
        "Mean Temp (°C)",
        "Total Rain (mm)",
        "Total Snow (cm)",
        "Total Precip (mm)"
    ]
].copy()

In [13]:
weather = weather.rename(
    columns={
        "Station Name": "station_name",

        "Max Temp (°C)":
            "max_temperature",

        "Min Temp (°C)":
            "min_temperature",

        "Mean Temp (°C)":
            "mean_temperature",

        "Total Rain (mm)":
            "total_rain",

        "Total Snow (cm)":
            "total_snow",

        "Total Precip (mm)":
            "total_precipitation"
    }
)

In [14]:
weather_cols = [
    "max_temperature",
    "min_temperature",
    "mean_temperature",
    "total_rain",
    "total_snow",
    "total_precipitation"
]

for col in weather_cols:

    weather[col] = pd.to_numeric(
        weather[col],
        errors="coerce"
    )

In [15]:
weather["has_temp"] = (
    weather["mean_temperature"]
    .notna()
    .astype(int)
)

weather["is_current_station"] = (
    weather["source_station_id"] == 51459
).astype(int)

weather = (
    weather
    .sort_values(
        [
            "date",
            "has_temp",
            "is_current_station"
        ]
    )
    .drop_duplicates(
        subset=["date"],
        keep="last"
    )
    .drop(
        columns=[
            "has_temp",
            "is_current_station"
        ]
    )
    .sort_values("date")
    .reset_index(drop=True)
)

In [16]:
print("Rows:", len(weather))

print(
    "Date range:",
    weather["date"].min(),
    "→",
    weather["date"].max()
)

print(
    "Unique dates:",
    weather["date"].nunique()
)

print(
    "Duplicate dates:",
    weather["date"].duplicated().sum()
)

display(weather.head())

Rows: 5844
Date range: 2011-01-01 00:00:00 → 2026-12-31 00:00:00
Unique dates: 5844
Duplicate dates: 0


,date,source_station_id,station_name,max_temperature,min_temperature,mean_temperature,total_rain,total_snow,total_precipitation
0,2011-01-01,5097,TORONTO LESTER B. PEARSON INT'L A,12.0,-1.3,5.4,8.6,0.0,8.6
1,2011-01-02,5097,TORONTO LESTER B. PEARSON INT'L A,-1.1,-5.5,-3.3,0.0,0.0,0.0
2,2011-01-03,5097,TORONTO LESTER B. PEARSON INT'L A,0.1,-6.5,-3.2,0.0,0.0,0.0
3,2011-01-04,5097,TORONTO LESTER B. PEARSON INT'L A,1.3,-3.2,-1.0,0.0,0.4,0.4
4,2011-01-05,5097,TORONTO LESTER B. PEARSON INT'L A,-1.0,-8.9,-5.0,0.0,0.0,0.0


In [18]:
# Missingness
missing = (
    weather
    .isna()
    .mean()
    .sort_values(ascending=False)
    * 100
)

display(
    missing.to_frame(
        "missing_pct"
    )
)

,missing_pct
total_rain,2.566735
mean_temperature,2.412731
total_snow,2.395619
min_temperature,2.378508
max_temperature,2.361396
total_precipitation,2.361396
date,0.000000
source_station_id,0.000000
station_name,0.000000


In [19]:
weather["year"] = weather["date"].dt.year
weather["month"] = weather["date"].dt.month
weather["day_of_year"] = weather["date"].dt.dayofyear

weather["is_rain"] = (
    weather["total_rain"].fillna(0) > 0
).astype("int8")

weather["is_snow"] = (
    weather["total_snow"].fillna(0) > 0
).astype("int8")

In [20]:
weather.to_parquet(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)

print(
    "Size:",
    round(
        OUTPUT_PATH.stat().st_size / 1024**2,
        2
    ),
    "MB"
)

Saved: data/external/weather/toronto_pearson_weather_daily.parquet
Size: 0.1 MB
